# 02 — Modelado y evaluación del Asistente de Delivery Económico

**Autor:** Ruben Vigil · **T13:** variables y separación temporal, ejecutable.
**T14:** baseline entrenado/evaluado. **T15:** tres familias entrenadas con train.
**T16:** validación/tuning temporal. **T17–T19:** pendientes. Los resultados T15 son errores de entrenamiento.

La cohorte real se preparó en el primer notebook. Aquí diseñamos el experimento,
entrenamos y registramos evidencias sin aprender de etiquetas futuras.


## Ejecución local o Colab

Localmente: `uv sync --locked --extra notebooks`, abrir este archivo y seleccionar
`.venv/bin/python`. En Google Colab: **Archivo → Subir notebook** y ejecutar las
celdas en orden, usando CPU. La primera celda pedirá el paquete actualizado:

```bash
uv run --locked --extra notebooks python ml/data/package_notebook_source.py
```

Seleccionar `data/reports/presentation/delivery_source.zip`. No contiene datos,
credenciales, `.env` ni `AGENTS.md`. El código no depende de esas instrucciones.
Las siguientes celdas adquieren/verifican el original y preparan la cohorte si no
está disponible: este cuaderno no depende de haber ejecutado el primero en la
misma máquina. No se monta Drive ni se usa AWS.

T13 usa Pandas y T14 añade scikit-learn; se ejecutan sus cinco pruebas. Las
pruebas Spark del producto completo siguen requiriendo Java 17. El entorno de Colab es
temporal; guardar el notebook no conserva por sí solo los CSV de su máquina.
[Referencia oficial](https://research.google.com/colaboratory/faq.html).

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile

# Local: buscar el proyecto desde el directorio del notebook o sus padres.
PROJECT_DIR = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "ml/data/prepare_meituan.py").is_file()
     and (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_DIR is None:
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError("Abrí este notebook dentro del repositorio o en Colab.") from error
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Seleccioná únicamente delivery_source.zip")
    archive_path = Path(next(iter(uploaded)))
    PROJECT_DIR = Path(tempfile.mkdtemp(prefix="delivery-colab-"))
    with zipfile.ZipFile(archive_path) as archive:
        source_manifest = json.loads(archive.read("SOURCE_MANIFEST.json"))
        for name, expected_hash in source_manifest["files"].items():
            target = (PROJECT_DIR / name).resolve()
            if PROJECT_DIR not in target.parents:
                raise ValueError("Ruta inválida en el paquete de fuentes")
            content = archive.read(name)
            if hashlib.sha256(content).hexdigest() != expected_hash:
                raise ValueError(f"Hash de fuente inválido: {name}")
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
    print("HEAD de referencia:", source_manifest["git_head"])
    print("Incluye cambios locales:", source_manifest["working_tree_changes"])
    print("Hashes del código verificados; el HEAD solo identifica la base.")
print("Proyecto:", PROJECT_DIR)

In [ ]:
# Aislar las dependencias del proyecto del kernel de Colab.
PROJECT_ENV = os.environ.copy()
PROJECT_ENV.pop("VIRTUAL_ENV", None)
uv_path = shutil.which("uv")
if uv_path:
    UV = [uv_path]
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "uv==0.12.23"], check=True
    )
    UV = [sys.executable, "-m", "uv"]
subprocess.run(UV + ["sync", "--locked", "--extra", "notebooks"], cwd=PROJECT_DIR, env=PROJECT_ENV, check=True)

def run_project(*arguments):
    result = subprocess.run(
        UV + ["run", "--locked", "--extra", "notebooks", *arguments],
        cwd=PROJECT_DIR, env=PROJECT_ENV, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    result.check_returncode()
    return result

def read_json(relative_path):
    return json.loads((PROJECT_DIR / relative_path).read_text(encoding="utf-8"))

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

from html import escape
from IPython.display import HTML, display

def show_table(headers, rows):
    head = "".join(f"<th>{escape(str(value))}</th>" for value in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

import runpy
REVISION = runpy.run_path(str(PROJECT_DIR / "ml/data/acquire_meituan.py"))["REVISION"]
RAW_DIR = Path("data/raw/meituan") / REVISION
AUDIT_PATH = Path("data/reports/meituan") / REVISION / "t07_audit.json"
PROCESSED_DIR = Path("data/processed/meituan") / REVISION / "t08-v1"

## Dependencias de datos — T06/T07/T08

El original se mantiene intacto. Los scripts verifican revisión y hashes; la
auditoría aprueba una cohorte de pedidos inmediatos completados. Que una etiqueta
sea observada no garantiza que el modelo vaya a ser útil fuera de esa cohorte.

In [ ]:
_ = run_project("python", "ml/data/acquire_meituan.py")
_ = run_project("python", "ml/data/audit_meituan.py")
_ = run_project("python", "ml/data/prepare_meituan.py")
prepared = read_json(PROCESSED_DIR / "manifest.json")
print("Cohorte disponible:", prepared["rows"], "pedidos; SHA-256:", prepared["sha256"])

## T13 — Variables disponibles al crear la solicitud

Usamos solo `request_hour_utc` y `request_weekday_utc`, derivados del timestamp
de solicitud. Convertimos cada uno en seno/coseno con períodos fijos de 24 horas
y 7 días. Así 23:00 y 00:00 permanecen próximos; no se aprende una media, escala
o vocabulario a partir del test.

Las cuatro columnas son la **lista explícita de predictores**. Los IDs y eventos
se conservan para trazabilidad/partición y nunca se incluyen automáticamente.
La llegada permite construir la etiqueta, pero es posterior a la cotización.
Coordenadas y distancia quedan fuera mientras sus unidades no estén confirmadas.

Esta versión exclusivamente temporal puede tener poca capacidad para distinguir
pedidos distintos. No se afirma que describa rutas o diferencias por vehículo.
T13 prepara un experimento académico válido; no promete calidad predictiva.

In [ ]:
_ = run_project("python", "-c", "import inspect; from delivery_ml.model_data import temporal_features; print(inspect.getsource(temporal_features))")

## Separar pasado y futuro, incluyendo disponibilidad de etiquetas

Se fija `2022-10-23T00:00:00Z`: las dos últimas fechas de creación UTC quedan
para prueba. Es una decisión del experimento basada en la cobertura temporal,
no una regla comercial ni una elección optimizada mirando métricas del test.

- `train`: solicitud y llegada anteriores al corte.
- `test`: solicitud desde el corte.
- `purged`: solicitud anterior pero llegada en/después del corte. Aún no existía
  esa etiqueta para entrenar en ese momento; se registra aparte.

No se impone una duración máxima ni se trasladan esos pedidos al test.
Se verifican IDs disjuntos, cobertura completa y llegada de entrenamiento
anterior al comienzo del test. Las particiones se congelan con receta y hashes;
cambiar el corte requiere otra versión de salida (`--output`).

[Evaluación temporal y fugas — scikit-learn](https://scikit-learn.org/stable/modules/cross_validation.html#time-series-split).

In [ ]:
_ = run_project("python", "ml/data/prepare_model_data.py")
SPLIT_DIR = Path("data/splits/meituan") / REVISION / "t13-temporal-v1"
split = read_json(SPLIT_DIR / "manifest.json")
assert split["recipe"]["source_sha256"] == prepared["sha256"]
assert sum(info["rows"] for info in split["partitions"].values()) == prepared["rows"]
assert all(split["checks"].values())
for info in split["partitions"].values():
    assert sha256_file(PROJECT_DIR / SPLIT_DIR / info["file"]) == info["sha256"]
show_table(["Partición", "Pedidos", "Creación UTC desde", "Hasta"], [
    [name, info["rows"], info["creation_min_utc"], info["creation_max_utc"]]
    for name, info in split["partitions"].items()
])
show_table(["Control", "Resultado"], split["checks"].items())
print("Features:", split["feature_columns"])
print("Target:", split["target"])
print("Entrenamiento:", split["training"])

## Entradas explícitas para los modelos futuros

`model_xy` selecciona únicamente la lista permitida y devuelve `X`/`y` por
separado. No usar «todas las columnas menos el target»: eso incluiría información
posterior como `arrive_time`. La siguiente celda solo inspecciona la forma del
entrenamiento, sin ajustar un modelo ni consultar estadísticas del target de test.

In [ ]:
inspection = "import json,sys; from pathlib import Path; import pandas as pd; from delivery_ml.model_data import model_xy; train=pd.read_csv(Path(sys.argv[1])/'train.csv'); x,y=model_xy(train); print(json.dumps({'X_train_shape':list(x.shape),'feature_columns':list(x.columns),'y_train_rows':len(y),'target':y.name},ensure_ascii=False,indent=2))"
_ = run_project("python", "-c", inspection, str(SPLIT_DIR))

In [ ]:
# Casos controlados solo de T13; no son datos de entrenamiento.
_ = run_project("pytest", "ml/tests/test_model_data.py", "-q")

## Límites de este diseño

El EDA T09 ya observó agregados de la cohorte completa. Por eso no describimos
este test como un experimento ciego a aquella exploración. Desde T13 se reserva
para evaluación: no ajustar features, escalado, hiperparámetros ni reglas con él.
Las transformaciones que aprendan estadísticas deberán ajustarse solo dentro
del entrenamiento y sus folds, cuando corresponda.

La cobertura es corta, con bordes parciales, datos de China y solo entregas
completadas inmediatas. No acredita desempeño en El Salvador, por vehículo o
para entregas fallidas. No se han confirmado unidades geográficas ni entrenado
un modelo. La versión temporal no cambia contratos HTTP de la aplicación.

[Preprocesamiento y fugas — scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

## T14 — Baseline de mediana de entrenamiento

**Objetivo:** tener una referencia sencilla antes de evaluar modelos complejos.
Se fija una sola estrategia, `DummyRegressor(strategy="median")`, antes de
consultar errores de test. Aprende la mediana de `y_train` y predice ese valor
para todos los pedidos, ignorando las variables. La mediana es coherente con
error absoluto; se reporta MAE como métrica principal y RMSE como diagnóstico.
No se comparan estrategias para elegir la que gane en test.

MAE es el promedio de errores absolutos, no la mediana de errores. RMSE aumenta
la penalización de errores grandes. Ambos se expresan en minutos, sin convertir
resultados a porcentajes de exactitud o promesas de entrega.

[Baseline oficial](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html),
[métricas de regresión](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics).

In [ ]:
_ = run_project("python", "-c", "import inspect; import delivery_ml.baseline as module; print(inspect.getsource(module))")

In [ ]:
_ = run_project("python", "ml/experiments/evaluate_baseline.py")
BASELINE_PATH = Path("data/reports/meituan") / REVISION / "t14_baseline.json"
baseline_report = read_json(BASELINE_PATH)
assert baseline_report["provenance"]["split_manifest_sha256"] == sha256_file(PROJECT_DIR / SPLIT_DIR / "manifest.json")
assert baseline_report["metrics"]["train"]["rows"] == split["partitions"]["train"]["rows"]
assert baseline_report["metrics"]["test"]["rows"] == split["partitions"]["test"]["rows"]
assert all(baseline_report["checks"].values())
show_table(["Modelo de referencia", "Valor"], [
    ["Estrategia fijada", baseline_report["strategy"]],
    ["Predicción constante (min)", baseline_report["constant_minutes"]],
    ["Origen de la constante", baseline_report["constant_source"]],
    ["Utiliza valores de features", baseline_report["uses_feature_values"]],
])
show_table(["Conjunto", "Pedidos", "MAE (min)", "RMSE (min)"], [
    [name, item["rows"], round(item["mae_minutes"], 6), round(item["rmse_minutes"], 6)]
    for name, item in baseline_report["metrics"].items()
])
print("Versión de software:", baseline_report["provenance"]["versions"])
for limitation in baseline_report["limitations"]:
    print("•", limitation)

In [ ]:
_ = run_project("pytest", "ml/tests/test_baseline.py", "-q")

## Interpretación

Esta referencia responde si un modelo futuro aporta información útil frente a
predecir una mediana histórica. Es una referencia de evaluación, con estrategia
fija; las decisiones de preprocesamiento/hiperparámetros posteriores se tomarán
solo en entrenamiento y validación interna. No ajustar el baseline ni elegir
modelos por las cifras del test.

Reejecutar esta receta reproduce la misma referencia; no constituye nuevas
pruebas independientes. El EDA global previo y las limitaciones de país/período
siguen vigentes. T19 decidirá persistencia del modelo final; no se sirve la API
ni se integra AWS en esta etapa.



## T15 — Tres algoritmos de familias distintas

- Regresión lineal: mínimos cuadrados.
- Random Forest: ensamble de árboles con bagging.
- LinearSVR: vectores de soporte con kernel lineal y pérdida epsilon-insensible.

Usamos todos los pedidos de train con las mismas cuatro variables. LinearSVR
permite mantener la cohorte completa con menor costo que SVR RBF; no es un
kernel no lineal. Los escaladores están dentro de los pipelines y solo se
ajustan con train. Las configuraciones son puntos de partida, sin tuning.

El script verifica hashes de todas las particiones, pero solo carga train.
No se evalúa test ni se persiste el modelo final en esta tarea.


In [ ]:
_ = run_project("python", "-c", "import inspect; import delivery_ml.models as module; print(inspect.getsource(module))")


In [ ]:
_ = run_project("python", "ml/experiments/train_models.py")
MODELS_PATH = Path("data/reports/meituan") / REVISION / "t15_models.json"
models_report = read_json(MODELS_PATH)
manifest_path = Path("data/splits/meituan") / REVISION / "t13-temporal-v1/manifest.json"
train_manifest = read_json(manifest_path)
assert models_report["provenance"]["split_manifest_sha256"] == sha256_file(PROJECT_DIR / manifest_path)
assert all(models_report["checks"].values())
assert len(models_report["models"]) == 3
assert all(item["training_rows"] == train_manifest["partitions"]["train"]["rows"] for item in models_report["models"].values())
show_table(["Modelo", "Pedidos train", "MAE train (min)", "RMSE train (min)"], [
    [name, item["training_rows"], round(item["training_mae_minutes"], 6), round(item["training_rmse_minutes"], 6)]
    for name, item in models_report["models"].items()
])
for name, item in models_report["models"].items():
    print(name, "—", item["family"], "— parámetros:", item["steps"])
for limitation in models_report["limitations"]:
    print("•", limitation)


In [ ]:
# Casos controlados de aislamiento; no son registros del experimento real.
_ = run_project("pytest", "ml/tests/test_models.py", "-q")


## Interpretar sin elegir un ganador todavía

Los errores de entrenamiento usan los mismos datos con que se ajustó el modelo:
pueden ser optimistas. No compararlos con los errores de test del baseline como
si procedieran del mismo conjunto. Tampoco afirmar generalización porque un
bosque tenga menor error aquí. La validación y búsqueda temporal interna son T16;
la comparación posterior entre modelos y baseline corresponde a T18.



## T16 — Validación temporal y búsqueda pequeña

Tres días de validación dentro de train, con entrenamiento creciente. Solo
entran pedidos cuya creación y entrega precedan estrictamente a cada corte.
No barajamos filas ni usamos el test reservado. Los escaladores se ajustan
dentro de los pipelines de cada fold.

Probamos profundidad 6/12 del bosque y C=0.1/1 en LinearSVR. Regresión lineal y
mediana son controles. Elegimos por MAE medio, igual peso por día; RMSE es
secundario. El elegido por familia se reajusta solo con todo train, en memoria.


In [ ]:
_ = run_project("python", "-c", "import inspect; import delivery_ml.validation as module; print(inspect.getsource(module))")


In [ ]:
_ = run_project("python", "ml/experiments/tune_models.py")
VALIDATION_PATH = Path("data/reports/meituan") / REVISION / "t16_validation.json"
validation_report = read_json(VALIDATION_PATH)
manifest_path = Path("data/splits/meituan") / REVISION / "t13-temporal-v1/manifest.json"
assert validation_report["provenance"]["split_manifest_sha256"] == sha256_file(PROJECT_DIR / manifest_path)
assert all(validation_report["checks"].values())
assert len(validation_report["folds"]) == 3
show_table(["Inicio validación UTC", "Filas train", "Filas validación", "Train purgado"], [
    [f["validation_start_utc"], f["training_rows"], f["validation_rows"], f["purged_training_rows"]]
    for f in validation_report["folds"]
])
show_table(["Modelo", "Parámetros elegidos", "MAE CV (min)", "RMSE CV (min)"], [
    [name, item["best_parameters"], round(item["best_mean_mae_minutes"], 6),
     round(item["candidates"][item["best_index"]]["mean_rmse_minutes"], 6)]
    for name, item in validation_report["models"].items()
])
for name, item in validation_report["models"].items():
    print(name, "— todos los candidatos y errores por fold:", item["candidates"])
print(validation_report["selection"])
for limitation in validation_report["limitations"]:
    print("•", limitation)


In [ ]:
# Casos controlados: purga temporal, escalado por fold y selección por MAE.
_ = run_project("pytest", "ml/tests/test_validation.py", "-q")


## Interpretación y pendientes

El bosque supera al baseline en estos folds internos. Como se usaron para
seleccionar, el MAE elegido puede ser optimista; no es un resultado independiente.
Tres días y dos valores por hiperparámetro son una búsqueda acotada, no un óptimo
global. El test continúa reservado y las limitaciones geográficas siguen vigentes.

**T17–T19 pendientes:** MLP, comparación posterior y selección/persistencia final.

Para exponer: ¿por qué no barajar entregas? ¿Por qué purgar una etiqueta real?
¿Qué diferencia MAE de entrenamiento, MAE de CV y MAE de test? ¿Por qué el escalado
se aprende de nuevo en cada fold? ¿Qué permite concluir esta búsqueda y qué no?
